# Overture motorways — interactive view

Fetches Overture `transportation` motorway segments (ramps dropped via the
`is_link` road flag) for the freeway-hex bbox using
`scripts/build_road_table.py`, loads them into a GeoDataFrame, and renders
with lonboard `viz`.

This is the road geometry that feeds the H3 columnar join into
`data/road_table.parquet`.

In [ ]:
import sys

sys.path.append("scripts")

import geopandas as gpd
import pyarrow.parquet as pq
from shapely import wkb

from overturemaps.core import get_latest_release
from build_road_table import MOTORWAYS, fetch_motorways, hex_bbox

bbox = hex_bbox()
release = get_latest_release()
print("release", release, "bbox", bbox)

# refetch=True re-pulls from Overture. The cached parquet is stale (pre-ramp-
# filter), so set True on the first run after changing the WHERE clause.
fetch_motorways(bbox, release, refetch=True)

release 2026-05-20.0 bbox (-124.78560603210607, 25.02821309446701, -67.3632084036063, 49.57700082876189)
[motorways] querying Overture 2026-05-20.0 within (-124.78560603210607, 25.02821309446701, -67.3632084036063, 49.57700082876189) ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
df = pq.read_table(MOTORWAYS).to_pandas()
df["geometry"] = df.geometry_wkb.map(lambda b: wkb.loads(bytes(b)))
gdf = gpd.GeoDataFrame(
    df[["road_id", "road_name"]],
    geometry=df.geometry,
    crs="EPSG:4326",
)
print(len(gdf), "motorway segments |", gdf.road_name.notna().sum(), "named")
gdf.head()

In [ ]:
from lonboard import viz

viz(gdf)